# Temporal graph benchmark exploration

Exploration only; this notebook is never imported by the application. It checks chronological split sizes and compares trained TGN/StrGNN artifacts with the configured statistical and rule baselines. Learned-model evaluation requires real saved weights.

In [ ]:
from pathlib import Path

from ml.temporal_graph.dataset import build_temporal_graph

cwd = Path.cwd()
parents = (cwd, *cwd.parents)
repo_root = next(
    (parent for parent in parents if (parent / 'data/processed').exists()),
    cwd,
)
data_root = repo_root / 'data/processed/sih_product/splits'
splits = {name: build_temporal_graph(data_root, name) for name in ('train', 'val', 'test')}
for name, data in splits.items():
    print(
        name, 'events=', len(data), 'sources=', data.num_sources,
        'entities=', data.num_entities, 'faults=', int(data.labels.sum())
    )

In [ ]:
assert splits['train'].timestamps.max() <= splits['val'].timestamps.min()
assert splits['val'].timestamps.max() <= splits['test'].timestamps.min()
print('Chronological partition boundaries verified.')

In [ ]:
from ml.temporal_graph.evaluate import evaluate_all

required_artifacts = [
    repo_root / 'ml/artifacts/tgn/model.pt',
    repo_root / 'ml/artifacts/strgnn/model.pt',
]
if all(path.is_file() for path in required_artifacts):
    results = evaluate_all('test')
    display(results)
else:
    print(
        'Train TGN and StrGNN artifacts before evaluating; no model scores are fabricated.'
    )